In [ ]:
# =========================================
# 📦 1. Setup & Data Loading
# =========================================

import pandas as pd
import openpyxl
from IPython.display import display

# Path to cleaned dataset (use relative path in production)
file_path = r"../data/edited/online_retail_cleaned.xlsx"

# Load dataset
df = pd.read_excel(file_path, engine='openpyxl')


# =========================================
# 👤 2. Customer-Level KPIs
# =========================================

customer_kpis = (
    df.groupby('CustomerID')
      .agg(
          TotalRevenue=('Revenue', 'sum'),
          Orders=('InvoiceNo', 'nunique'),
          TotalItems=('Quantity', 'sum'),
          FirstPurchase=('InvoiceDate', 'min'),
          LastPurchase=('InvoiceDate', 'max')
      )
      .reset_index()
)

customer_kpis.head()


# =========================================
# 💰 3. Average Order Value (AOV)
# =========================================

customer_kpis['AOV'] = customer_kpis['TotalRevenue'] / customer_kpis['Orders']

customer_kpis.describe()


# =========================================
# 🔁 4. Customer Segmentation (Basic)
# =========================================

# Classify customers based on purchase frequency
customer_kpis['CustomerType'] = customer_kpis['Orders'].apply(
    lambda x: 'One-time' if x == 1 else 'Repeat'
)

# Distribution of customer types
customer_kpis['CustomerType'].value_counts(normalize=True) * 100


# =========================================
# 📅 5. December Analysis (Seasonality Case)
# =========================================

df['Month'] = df['InvoiceDate'].dt.month

december_df = df[df['Month'] == 12]

# Customer performance in December
december_customers = (
    december_df.groupby('CustomerID')
      .agg(
          DecemberRevenue=('Revenue', 'sum'),
          DecemberOrders=('InvoiceNo', 'nunique')
      )
      .reset_index()
)

december_customers.sort_values('DecemberRevenue', ascending=False).head(10)


# =========================================
# 📊 6. December AOV by Customer Type
# =========================================

december_customer_type_aov = (
    december_df
    .merge(customer_kpis[['CustomerID', 'CustomerType']], on='CustomerID')
    .groupby('CustomerType')
    .agg(
        Revenue=('Revenue', 'sum'),
        Orders=('InvoiceNo', 'nunique')
    )
    .reset_index()
)

december_customer_type_aov['AOV'] = (
    december_customer_type_aov['Revenue'] / december_customer_type_aov['Orders']
)

december_customer_type_aov


# =========================================
# 📊 7. Full-Year AOV by Customer Type
# =========================================

customer_type_aov = (
    df
    .merge(customer_kpis[['CustomerID', 'CustomerType']], on='CustomerID')
    .groupby('CustomerType')
    .agg(
        Revenue=('Revenue', 'sum'),
        Orders=('InvoiceNo', 'nunique')
    )
    .reset_index()
)

customer_type_aov['AOV'] = (
    customer_type_aov['Revenue'] / customer_type_aov['Orders']
)

display(december_customer_type_aov)
customer_type_aov


# =========================================
# 💸 8. Revenue Concentration Analysis
# =========================================

# Top customers by revenue
display(customer_kpis.sort_values('TotalRevenue', ascending=False).head(10))

# Contribution of top 10 customers
customer_kpis.sort_values('TotalRevenue', ascending=False).head(10)['TotalRevenue'].sum() / customer_kpis['TotalRevenue'].sum()